[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [6]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [8]:
import torch
import torch.nn as nn
import math

In [14]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        # pass  # W_q, W_k, W_v, W_o

    def forward(self, x_q, x_kv):
        Q = self.W_q(x_q) # (b, s, d) -> (b, h, s, h_d)
        Q: torch.Tensor = Q.reshape(-1, Q.shape[1], self.num_heads, self.head_dim).transpose(1,2) # (b, h, s_q, h_d)
        K = self.W_k(x_kv)
        K = K.reshape(-1, K.shape[1], self.num_heads, self.head_dim).transpose(1,2) # (b, h, s_kv, h_d)
        V = self.W_v(x_kv)
        V = V.reshape(-1, V.shape[1], self.num_heads, self.head_dim).transpose(1,2) # (b, h, s_kv, h_d)

        # now the cross-attention
        # Q@K^T
        attn_logits = torch.einsum("ijmk,ijnk->ijmn", Q, K) # (b, h, s_q, s_kv)
        attn_logits = attn_logits / math.sqrt(self.head_dim)
        attn_scores = nn.functional.softmax(attn_logits, dim=-1)
        attn_output = torch.einsum("ijmn,ijnk->imjk", attn_scores, V) # (b, h, s_kv, h_d)
        attn_output = attn_output.reshape(-1, x_q.shape[1], self.d_model)

        # final projection
        output = self.W_o(attn_output)
        return output



        pass  # Q from x_q, K/V from x_kv, no causal mask

In [15]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [16]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.0ms)
  ✅ [2/4] Q and KV different lengths (0.9ms)
  ✅ [3/4] No causal mask — all KV affects all Q (44.4ms)
  ✅ [4/4] Gradient flow (25.8ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (73.2ms total)
  Progress saved. Run status() to see your dashboard.

